# Default Baseline Models Pipeline

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error

import pmdarima as pm
from arch import arch_model
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import het_arch

In [2]:
# config

OUT_DIR = Path(r"C:\\Users\\user\\OneDrive\\Desktop\\Simulations\\processed")

N_LAGS = 10
STEP = 1
TEST_SIZE = 1

MIN_SAMPLES_FOR_CV = 20
MIN_SAMPLES_FOR_ARIMA = 30

In [4]:
# metrics

def directional_accuracy(y_true, y_pred):
    return np.mean(np.sign(np.diff(y_true)) ==
                   np.sign(np.diff(y_pred))
    )

def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / 
                         (np.abs(y_true) + np.abs(y_pred) + 1e-12))



In [5]:
# Walk-Forwad prediction

def walk_forward_predict(model_func, X_train, y_train, X_test, y_test, step=STEP):
    preds = []
    history_X, history_y = X_train.copy(), y_train.copy()

    for i in range(0, len(X_test), step):

        # NEW: dont build models when history is too small.
        if len(history_y) < 10:
            # skip early windows, nt enough data
            preds.append(y_train.iloc[-1]) # fallback = last value
            history_X = pd.concat([history_X, X_test.iloc[i:i+step]])
            history_y = pd.concat([history_y, y_test.iloc[i:i+step]])
            continue


        model = model_func(history_X, history_y)

        X_next = X_test.iloc[i:i+step]
        y_pred = model.predict(X_next)
        
        preds.extend(y_pred)
                   
        # expanding window
        history_X = pd.concat([history_X, X_next])
        history_y = pd.concat([history_y, y_test.iloc[i:i+step]])
    
    return np.array(preds)

In [6]:
# traditional Ml Baseline models

def train_lr(X, y):
    model = LinearRegression()
    return model.fit(X, y)

def train_rf(X, y):
    if len(X) < MIN_SAMPLES_FOR_CV:
        # fallback - no CV
        return RandomForestRegressor().fit(X, y)
    
    param_grid = {
        'n_estimators': [50, 100],
        'max_depth': [3, 5, None]
    }
    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(
        RandomForestRegressor(), 
        param_grid, 
        cv=tscv, 
        scoring='neg_mean_squared_error'
    )
    grid.fit(X, y)
    return grid.best_estimator_

def train_mlp(X, y):
    if len(X) < MIN_SAMPLES_FOR_CV:
        return MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=500).fit(X, y)
    
    param_grid = {
        'hidden_layer_sizes': [(32,), (64,), (32, 16)],
        'alpha': [0.0001, 0.001],
    }

    tscv = TimeSeriesSplit(n_splits=3)
    grid = GridSearchCV(
        MLPRegressor(max_iter=500), 
        param_grid, 
        cv=tscv, 
        scoring='neg_mean_squared_error'
    )
    grid.fit(X, y)
    return grid.best_estimator_

In [12]:
# Automatic ARIMA/SARIMA/GARCH selector

def time_series_model(X, y):

    series = y.values


    # too short --> retun simple fallback
    if len(series) < MIN_SAMPLES_FOR_ARIMA:
        class NaiveMean:
            def __init__(self, last_value):
                self.val = last_value

            def predict(self, Xn):
                n = len(Xn)
                return np.repeat(self.val, n)

        return NaiveMean(series[-1])

    # ACF seasonality detection
    try:
        ac = acf(series, nlags=min(50, len(series)//2))
        seasonal_lags = np.where(ac > 0.5)[0]
        has_seasonality = len(seasonal_lags) > 1 and seasonal_lags[1] > 1
        m = seasonal_lags[1] if has_seasonality else 1
    except Exception:
        has_seasonality = False
        m = 1

    # ARCH test
    try:
        arch_stat, arch_pval, _, _ = het_arch(series)
        is_heteroskedastic = arch_pval < 0.05
    except Exception:
        is_heteroskedastic = False


    # GARCH model

    if is_heteroskedastic:
        garch = arch_model(series, vol='Garch', p=1, q=1)
        fitted = garch.fit(disp='off')

        class GarchWrapper:
            def __init__(self, model): self.m = model
            def predict(self, Xn): 
                fc = self.m.forecast(horizon=len(Xn)).mean.values
                return fc[-1] # corrected shape
        return GarchWrapper(fitted)
    
    # SARIMA mdel

    if has_seasonality:
        model = pm.auto_arima(series, seasonal=True, m=m,
                              stepwise=True, suppress_warnings=True)
        
        class SarimaWrapper:
            def __init__(self, model): self.m = model
            def predict(self, Xn): return self.m.predict(n_periods=len(Xn))

        return SarimaWrapper(model)
    
    # ARIMA model
    
    model = pm.auto_arima(series, seasonal=False,
                              stepwise=True, suppress_warnings=True)
        
    class ArimaWrapper:
        def __init__(self, model): self.m = model
        def predict(self, Xn): return self.m.predict(n_periods=len(Xn))

    return ArimaWrapper(model)
        

In [13]:
### Run Pipeline

for process_path in OUT_DIR.iterdir():
    if not process_path.is_dir():
        continue

    metrics_table = []

    print(f"\n=== Baseline Models for {process_path.name} ===")

    for train_file in process_path.glob("tabular_train_step*.csv"):
        
        step_idx = train_file.stem.split("step")[-1]

        df_train = pd.read_csv(train_file)
        df_test_file = process_path / f"tabular_test_step{step_idx}.csv"
        if not df_test_file.exists():
            continue

        df_test = pd.read_csv(df_test_file)

        feature_cols = [f"lag_{i}" for i in range (1, N_LAGS + 1)]
        X_train, y_train = df_train[feature_cols], df_train["y"]
        X_test, y_test = df_test[feature_cols], df_test["y"]

        models = {
            "Ridge": train_lr,
            "RandomForest": train_rf,
            "MLP": train_mlp,
            "TSModel(Auto)": time_series_model
        }

        step_results = {}

        # Skip small early training windows

        if len(X_train) < 10:
            continue

        for name, func in models.items():
            print(f" Running {name} (step {step_idx})...")

            preds = walk_forward_predict(func, X_train, y_train, X_test, y_test)

            mse = mean_squared_error(y_test, preds)
            mae = mean_absolute_error(y_test, preds)
            rmse = np.sqrt(mse)
            da = directional_accuracy(y_test.values, preds)
            sm = smape(y_test.values, preds)

            step_results[name] = {
                "forecast": preds,
                "mse": mse,
                "mae": mae,
                "rmse": rmse,
                "da": da,
                "smape": sm
            }

            # store row for CSV
            metrics_table.append({
                "process": process_path.name,
                "step": step_idx,
                "model": name,
                "MSE": mse,
                "MAE": mae,
                "RMSE": rmse,
                "Da": da,
                "SMAPE": sm
            })

            joblib.dump(step_results, process_path / f"baseline_results_step{step_idx}.joblib")
            print(f" Saved baseline results for step {step_idx}")


    # save metrics CSV per process
    df_metrics = pd.DataFrame(metrics_table)
    df_metrics.to_csv(process_path / "baseline_metrics.csv", index=False)
    print(f"Saved metrics CSV for {process_path.name}")


=== Baseline Models for process1 ===
 Running Ridge (step 10)...
 Saved baseline results for step 10
 Running RandomForest (step 10)...
 Saved baseline results for step 10
 Running MLP (step 10)...
 Saved baseline results for step 10
 Running TSModel(Auto) (step 10)...
 Saved baseline results for step 10
 Running Ridge (step 100)...
 Saved baseline results for step 100
 Running RandomForest (step 100)...


c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\user\py

 Saved baseline results for step 100
 Running MLP (step 100)...


c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


 Saved baseline results for step 100
 Running TSModel(Auto) (step 100)...
 Saved baseline results for step 100
 Running Ridge (step 101)...
 Saved baseline results for step 101
 Running RandomForest (step 101)...


...

c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


 Saved baseline results for step 99
 Running TSModel(Auto) (step 99)...
 Saved baseline results for step 99
 Running Ridge (step 990)...
 Saved baseline results for step 990
 Running RandomForest (step 990)...


c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


 Saved baseline results for step 990
 Running MLP (step 990)...
 Saved baseline results for step 990
 Running TSModel(Auto) (step 990)...
 Saved baseline results for step 990
Saved metrics CSV for process3


c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
c:\Users\user\python10\lib\site-packages\numpy\_core\fromnumeric.py:3860: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\user\python10\lib\site-packages\numpy\_core\_methods.py:145: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
